# PlantUML Parser Demonstration

This notebook demonstrates the PlantUML parser and helps you understand how it converts PlantUML text into structured Python objects.

## Table of Contents
1. [Setup](#1-setup)
2. [Basic Usage](#2-basic-usage)
3. [Understanding the Data Model](#3-understanding-the-data-model)
4. [Parsing Examples](#4-parsing-examples)
5. [Dataset Explorer](#5-dataset-explorer)
6. [Interactive Parsing](#6-interactive-parsing)

## 1. Setup

First, let's import the necessary modules.

In [1]:
import sys
import json
from pathlib import Path

# Add project root to path
NOTEBOOK_DIR = Path().absolute()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import parser components
from TestingMetrics.dummyMetric.Parser import (
    PlantUMLParser,
    ParsedModel,
    ParsedClass,
    ParsedEnum,
    ParsedRelationship,
    ParsedAttribute,
)
from TestingMetrics.dummyMetric.Parser.models import RelationshipType

print("Parser modules loaded successfully!")
print(f"Project root: {PROJECT_ROOT}")

Parser modules loaded successfully!
Project root: /mnt/c/Users/vasse/diss-metrik/Developing-DISS-Metric


## 2. Basic Usage

The parser takes a PlantUML string and returns a `ParsedModel` object containing all the extracted elements.

In [2]:
# Create a parser instance
parser = PlantUMLParser(strict=True)

# A simple PlantUML example
simple_uml = """
@startuml

class Person {
    name : String
    age : Integer
}

class Address {
    street
    city
}

Person "1" -- "*" Address : livesAt

@enduml
"""

# Parse it
model = parser.parse(simple_uml)

# Display the summary
print(model.summary())

ParsedModel: 2 classes, 0 enums, 1 relationships, 0 implicit classes


In [3]:
# Let's explore what was parsed
print("=" * 50)
print("CLASSES")
print("=" * 50)
for cls in model.classes:
    print(f"\nClass: {cls.name} (abstract: {cls.is_abstract})")
    for attr in cls.attributes:
        type_str = f" : {attr.type}" if attr.type else ""
        print(f"  - {attr.name}{type_str}")

CLASSES

Class: Person (abstract: False)
  - name : String
  - age : Integer

Class: Address (abstract: False)
  - street
  - city


In [4]:
print("=" * 50)
print("RELATIONSHIPS")
print("=" * 50)
for rel in model.relationships:
    print(f"\n{rel.source} -> {rel.target}")
    print(f"  Type: {rel.relationship_type.value}")
    print(f"  Source cardinality: {rel.source_cardinality}")
    print(f"  Target cardinality: {rel.target_cardinality}")
    print(f"  Label: {rel.label}")

RELATIONSHIPS

Person -> Address
  Type: association
  Source cardinality: 1
  Target cardinality: *
  Label: livesAt


## 3. Understanding the Data Model

The parser produces a `ParsedModel` containing:

- **classes**: List of `ParsedClass` objects (explicitly defined classes)
- **enums**: List of `ParsedEnum` objects (standalone enumerations)
- **relationships**: List of `ParsedRelationship` objects
- **notes**: List of `ParsedNote` objects
- **implicit_classes**: List of class names only mentioned in relationships

### Data Class Hierarchy

```
ParsedModel
├── classes: List[ParsedClass]
│   ├── name: str
│   ├── is_abstract: bool
│   ├── attributes: List[ParsedAttribute]
│   │   ├── name: str
│   │   ├── type: Optional[str]
│   │   ├── default_value: Optional[str]
│   │   └── is_constant: bool
│   └── nested_enums: List[ParsedEnum]
├── enums: List[ParsedEnum]
│   ├── name: str
│   ├── values: List[str]
│   └── is_inline: bool
├── relationships: List[ParsedRelationship]
│   ├── source: str
│   ├── target: str
│   ├── relationship_type: RelationshipType
│   ├── source_cardinality: Optional[str]
│   ├── target_cardinality: Optional[str]
│   ├── label: Optional[str]
│   └── association_members: Optional[tuple]
└── implicit_classes: List[str]
```

In [5]:
# Let's see all the relationship types supported
print("Supported Relationship Types:")
print("-" * 40)
for rt in RelationshipType:
    print(f"  {rt.name}: {rt.value}")

Supported Relationship Types:
----------------------------------------
  ASSOCIATION: association
  DIRECTED_ASSOCIATION: directed
  INHERITANCE: inheritance
  COMPOSITION: composition
  AGGREGATION: aggregation
  DEPENDENCY: dependency
  ASSOCIATION_CLASS: association_class


## 4. Parsing Examples

Let's see how the parser handles different PlantUML syntax patterns.

### 4.1 Enumerations

In [6]:
enum_uml = """
@startuml

' Multi-line enum
enum Status {
    Active
    Inactive
    Pending
}

' Inline enum
enum Color { Red, Green, Blue }

@enduml
"""

model = parser.parse(enum_uml)

print("Parsed Enums:")
for enum in model.enums:
    inline_str = " (inline)" if enum.is_inline else ""
    print(f"\n  {enum.name}{inline_str}")
    print(f"    Values: {', '.join(enum.values)}")

Parsed Enums:

  Status
    Values: Active, Inactive, Pending

  Color (inline)
    Values: Red, Green, Blue


### 4.2 Inheritance

In [7]:
inheritance_uml = """
@startuml

' Standard inheritance (parent on left)
Animal <|-- Dog
Animal <|-- Cat

' Reverse notation (child on left)
Bird --|> Animal

@enduml
"""

model = parser.parse(inheritance_uml)

print("Parsed Inheritance Relationships:")
print("(Parser normalizes so parent is always 'source')")
print()
for rel in model.relationships:
    print(f"  {rel.source} <|-- {rel.target}")
    print(f"    (Parent: {rel.source}, Child: {rel.target})")

Parsed Inheritance Relationships:
(Parser normalizes so parent is always 'source')

  Animal <|-- Dog
    (Parent: Animal, Child: Dog)
  Animal <|-- Cat
    (Parent: Animal, Child: Cat)
  Animal <|-- Bird
    (Parent: Animal, Child: Bird)


### 4.3 Composition and Aggregation

In [8]:
comp_agg_uml = """
@startuml

' Composition (filled diamond) - strong ownership
Car "1" *-- "4" Wheel
House "1" *-- "*" Room

' Aggregation (hollow diamond) - weak ownership
Department "1" o-- "*" Employee
Library "1" o-- "*" Book

@enduml
"""

model = parser.parse(comp_agg_uml)

print("Parsed Composition/Aggregation:")
print("(Parser normalizes so whole/composite is always 'source')")
print()
for rel in model.relationships:
    arrow = "*--" if rel.relationship_type == RelationshipType.COMPOSITION else "o--"
    src_card = f'"{rel.source_cardinality}" ' if rel.source_cardinality else ""
    tgt_card = f' "{rel.target_cardinality}"' if rel.target_cardinality else ""
    print(f"  {rel.source} {src_card}{arrow}{tgt_card} {rel.target}")
    print(f"    Type: {rel.relationship_type.value}")

Parsed Composition/Aggregation:
(Parser normalizes so whole/composite is always 'source')

  Car "1" *-- "4" Wheel
    Type: composition
  House "1" *-- "*" Room
    Type: composition
  Department "1" o-- "*" Employee
    Type: aggregation
  Library "1" o-- "*" Book
    Type: aggregation


### 4.4 Association Classes

In [9]:
assoc_class_uml = """
@startuml

class Student {}
class Course {}
class Enrollment {
    grade
    enrollmentDate
}

' Association class linking Student and Course
(Student, Course) .. Enrollment

@enduml
"""

model = parser.parse(assoc_class_uml)

print("Parsed Association Class:")
for rel in model.relationships:
    if rel.relationship_type == RelationshipType.ASSOCIATION_CLASS:
        print(f"  Association class: {rel.target}")
        print(f"  Links: {rel.association_members[0]} and {rel.association_members[1]}")

Parsed Association Class:
  Association class: Enrollment
  Links: Student and Course


### 4.5 Nested Enums Inside Classes

In [10]:
nested_enum_uml = """
@startuml

class Device {
    enum DeviceStatus { Activated, Deactivated }
    DeviceStatus status
    Integer deviceID
}

@enduml
"""

model = parser.parse(nested_enum_uml)

print("Parsed Class with Nested Enum:")
for cls in model.classes:
    print(f"\nClass: {cls.name}")
    print("  Attributes:")
    for attr in cls.attributes:
        type_str = f" : {attr.type}" if attr.type else ""
        print(f"    - {attr.name}{type_str}")
    print("  Nested Enums:")
    for enum in cls.nested_enums:
        print(f"    - {enum.name}: {', '.join(enum.values)}")

Parsed Class with Nested Enum:

Class: Device
  Attributes:
    - status : DeviceStatus
    - deviceID : Integer
  Nested Enums:
    - DeviceStatus: Activated, Deactivated


### 4.6 Implicit Classes

In [11]:
implicit_uml = """
@startuml

' Only Order is explicitly defined
class Order {
    orderNumber
    date
}

' Customer and Product are only mentioned in relationships
Customer "1" -- "*" Order
Order "*" -- "*" Product

@enduml
"""

model = parser.parse(implicit_uml)

print("Explicit Classes:")
for cls in model.classes:
    print(f"  - {cls.name}")

print("\nImplicit Classes (only in relationships):")
for name in model.implicit_classes:
    print(f"  - {name}")

print("\nAll Class Names:")
print(f"  {model.all_class_names}")

Explicit Classes:
  - Order

Implicit Classes (only in relationships):
  - Customer
  - Product

All Class Names:
  ['Customer', 'Order', 'Product']


## 5. Dataset Explorer

Now let's load the actual dataset and explore real PlantUML examples.

In [12]:
# Load the dataset
DATASET_PATH = PROJECT_ROOT / "Dataset" / "combined-data.json"

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

models = dataset['models']
print(f"Loaded {len(models)} domain models:")
for key, data in models.items():
    print(f"  - {key}: {data['full_name']}")

Loaded 8 domain models:
  - LabTracker: Lab Requisition Management System
  - CelO: Celebrations Organization System (CelO)
  - TSS: Team Sports Scouting (TSS)
  - SHAS: SHAS
  - OTS: Online Tutoring System (OTS)
  - Block: Block  
  - TileO: Tile-O
  - HBMS: Hotel Booking Management System (HBMS)


In [13]:
# Build a list of all available PlantUML strings
SETTINGS = ['reference', '0shot', '1shot_BTMS', '1shot_H2S', '2shots', 'CoT']

all_examples = []
for model_key, model_data in models.items():
    # Reference model
    ref_uml = model_data.get('reference_plantuml', '')
    if ref_uml:
        all_examples.append({
            'id': f"{model_key}_reference",
            'model': model_key,
            'setting': 'reference',
            'uml': ref_uml,
            'is_reference': True
        })
    
    # Generated models
    generated = model_data.get('generated_plantuml', {})
    for setting in ['0shot', '1shot_BTMS', '1shot_H2S', '2shots', 'CoT']:
        gen_uml = generated.get(setting, '')
        if gen_uml:
            all_examples.append({
                'id': f"{model_key}_{setting}",
                'model': model_key,
                'setting': setting,
                'uml': gen_uml,
                'is_reference': False
            })

print(f"Total PlantUML examples available: {len(all_examples)}")
print("\nExample IDs:")
for i, ex in enumerate(all_examples):
    print(f"  [{i:2d}] {ex['id']}")

Total PlantUML examples available: 47

Example IDs:
  [ 0] LabTracker_reference
  [ 1] LabTracker_0shot
  [ 2] LabTracker_1shot_BTMS
  [ 3] LabTracker_1shot_H2S
  [ 4] LabTracker_2shots
  [ 5] LabTracker_CoT
  [ 6] CelO_reference
  [ 7] CelO_0shot
  [ 8] CelO_1shot_BTMS
  [ 9] CelO_1shot_H2S
  [10] CelO_2shots
  [11] CelO_CoT
  [12] TSS_reference
  [13] TSS_0shot
  [14] TSS_1shot_BTMS
  [15] TSS_1shot_H2S
  [16] TSS_2shots
  [17] TSS_CoT
  [18] SHAS_reference
  [19] SHAS_1shot_BTMS
  [20] SHAS_1shot_H2S
  [21] SHAS_2shots
  [22] SHAS_CoT
  [23] OTS_reference
  [24] OTS_0shot
  [25] OTS_1shot_BTMS
  [26] OTS_1shot_H2S
  [27] OTS_2shots
  [28] OTS_CoT
  [29] Block_reference
  [30] Block_0shot
  [31] Block_1shot_BTMS
  [32] Block_1shot_H2S
  [33] Block_2shots
  [34] Block_CoT
  [35] TileO_reference
  [36] TileO_0shot
  [37] TileO_1shot_BTMS
  [38] TileO_1shot_H2S
  [39] TileO_2shots
  [40] TileO_CoT
  [41] HBMS_reference
  [42] HBMS_0shot
  [43] HBMS_1shot_BTMS
  [44] HBMS_1shot_H2S
  [45

### Helper Function for Pretty Printing

In [14]:
def display_parsed_model(model: ParsedModel, show_raw: bool = False, raw_uml: str = None):
    """
    Display a parsed model in a readable format.
    
    Args:
        model: The ParsedModel to display
        show_raw: Whether to show the original PlantUML
        raw_uml: The original PlantUML string (if show_raw=True)
    """
    if show_raw and raw_uml:
        print("=" * 70)
        print(" ORIGINAL PLANTUML")
        print("=" * 70)
        for i, line in enumerate(raw_uml.strip().split('\n'), 1):
            print(f"{i:3d} | {line}")
        print()
    
    print("=" * 70)
    print(" PARSED MODEL")
    print("=" * 70)
    
    # Summary
    print(f"\n{model.summary()}")
    
    # Classes
    print(f"\n[CLASSES] ({len(model.classes)} explicit, {len(model.implicit_classes)} implicit)")
    for cls in model.classes:
        prefix = "abstract " if cls.is_abstract else ""
        print(f"  {prefix}class {cls.name}")
        for attr in cls.attributes:
            attr_str = f"    - {attr.name}"
            if attr.type:
                attr_str += f" : {attr.type}"
            if attr.default_value:
                attr_str += f" = {attr.default_value}"
            if attr.is_constant:
                attr_str = f"    - const {attr.name} : {attr.type} = {attr.default_value}"
            print(attr_str)
        for nested in cls.nested_enums:
            print(f"    - enum {nested.name} {{ {', '.join(nested.values)} }}")
    
    if model.implicit_classes:
        print(f"  (implicit: {', '.join(model.implicit_classes)})")
    
    # Enums
    if model.enums:
        print(f"\n[ENUMS] ({len(model.enums)} standalone)")
        for enum in model.enums:
            inline = " (inline)" if enum.is_inline else ""
            print(f"  enum {enum.name}{inline}: {', '.join(enum.values)}")
    
    # Relationships
    print(f"\n[RELATIONSHIPS] ({len(model.relationships)} total)")
    
    # Group by type for better readability
    by_type = {}
    for rel in model.relationships:
        rt = rel.relationship_type.value
        if rt not in by_type:
            by_type[rt] = []
        by_type[rt].append(rel)
    
    arrow_map = {
        'association': '--',
        'directed': '-->',
        'inheritance': '<|--',
        'composition': '*--',
        'aggregation': 'o--',
        'dependency': '..',
        'association_class': '..',
    }
    
    for rel_type, rels in by_type.items():
        print(f"  [{rel_type}]")
        for rel in rels:
            src_card = f' "{rel.source_cardinality}"' if rel.source_cardinality else ''
            tgt_card = f' "{rel.target_cardinality}"' if rel.target_cardinality else ''
            label = f' : {rel.label}' if rel.label else ''
            arrow = arrow_map.get(rel_type, '--')
            
            if rel.association_members:
                src = f"({rel.association_members[0]}, {rel.association_members[1]})"
            else:
                src = rel.source
            
            print(f"    {src}{src_card} {arrow}{tgt_card} {rel.target}{label}")
    
    # Notes
    if model.notes:
        print(f"\n[NOTES] ({len(model.notes)} total)")
        for note in model.notes:
            preview = note.content[:50] + "..." if len(note.content) > 50 else note.content
            print(f"  - {note.position}: {preview}")

print("Helper function 'display_parsed_model' defined.")

Helper function 'display_parsed_model' defined.


### Select and Parse a Dataset Example

Change the `EXAMPLE_INDEX` variable to explore different examples.

In [15]:
# ============================================
# CHANGE THIS INDEX TO EXPLORE DIFFERENT EXAMPLES
# ============================================
EXAMPLE_INDEX = 0  # 0 = LabTracker_reference

# You can also use negative indices:
# -1 = last example (HBMS_CoT)
# ============================================

example = all_examples[EXAMPLE_INDEX]

print(f"Selected Example: [{EXAMPLE_INDEX}] {example['id']}")
print(f"Model: {example['model']}")
print(f"Setting: {example['setting']}")
print(f"Is Reference: {example['is_reference']}")
print(f"Lines: {len(example['uml'].splitlines())}")

Selected Example: [0] LabTracker_reference
Model: LabTracker
Setting: reference
Is Reference: True
Lines: 107


In [16]:
# Parse and display the selected example
parsed = parser.parse(example['uml'])
display_parsed_model(parsed, show_raw=True, raw_uml=example['uml'])

 ORIGINAL PLANTUML
  1 | @startuml
  2 | 
  3 | enum Interval {
  4 |     weekly
  5 |     monthly
  6 |     everyHalfYear
  7 |     yearly
  8 | }
  9 | 
 10 | enum AccessType {
 11 |     reservable
 12 |     walkIn
 13 |     dropOff
 14 | }
 15 | 
 16 | enum DayOfWeek {
 17 |     Monday
 18 |     Tuesday
 19 |     Wednesday
 20 |     Thursday
 21 |     Friday
 22 |     Saturday
 23 |     Sunday
 24 | }
 25 | 
 26 | abstract class PersonRole {
 27 |     idNumber
 28 | }
 29 | 
 30 | class Person {
 31 |     lastName
 32 |     firstName
 33 |     address
 34 |     phoneNumber
 35 | }
 36 | 
 37 | class Patient {
 38 |     dateOfBirth
 39 | }
 40 | 
 41 | class Doctor {
 42 |     signature
 43 | }
 44 | 
 45 | class Requisition {
 46 |     effectiveDate
 47 |     repetitionCount
 48 |     repetitionInterval : Interval
 49 | }
 50 | 
 51 | class TestResult {
 52 |     negative
 53 |     report
 54 | }
 55 | 
 56 | class SpecificTest {
 57 |     date
 58 | }
 59 | 
 60 | class Appointment

### Quick Access Functions

In [ ]:
def parse_by_id(example_id: str):
    """
    Parse a dataset example by its ID.
    
    Example IDs: 'LabTracker_reference', 'CelO_0shot', 'SHAS_CoT', etc.
    """
    for ex in all_examples:
        if ex['id'] == example_id:
            parsed = parser.parse(ex['uml'])
            display_parsed_model(parsed, show_raw=True, raw_uml=ex['uml'])
            return parsed
    print(f"Example '{example_id}' not found.")
    return None

def parse_by_model_setting(model_key: str, setting: str = 'reference'):
    """
    Parse a dataset example by model key and setting.
    
    Args:
        model_key: 'LabTracker', 'CelO', 'TSS', 'SHAS', 'OTS', 'Block', 'TileO', 'HBMS'
        setting: 'reference', '0shot', '1shot_BTMS', '1shot_H2S', '2shots', 'CoT'
    """
    example_id = f"{model_key}_{setting}"
    return parse_by_id(example_id)

print("Quick access functions defined:")
print("  - parse_by_id('LabTracker_reference')")
print("  - parse_by_model_setting('CelO', '0shot')")

In [ ]:
# Try it out! Uncomment one of these:

# parse_by_id('LabTracker_reference')
# parse_by_model_setting('CelO', '0shot')
# parse_by_model_setting('SHAS', 'reference')  # Has nested enums!
# parse_by_model_setting('HBMS', 'CoT')

## 6. Interactive Parsing

You can also parse your own PlantUML strings.

In [ ]:
# Write your own PlantUML here:
my_uml = """
@startuml

' Your PlantUML goes here
class Example {
    attribute1 : String
    attribute2
}

class Related {}

Example "1" --> "*" Related : has

@enduml
"""

# Parse it
my_parsed = parser.parse(my_uml)
display_parsed_model(my_parsed, show_raw=True, raw_uml=my_uml)

### Non-Strict Mode

By default, the parser is in strict mode and will raise an error for unrecognized syntax. You can use non-strict mode to skip unrecognized lines with warnings.

In [ ]:
# Non-strict parser
lenient_parser = PlantUMLParser(strict=False)

uml_with_errors = """
@startuml

class ValidClass {}

this line is not valid PlantUML

ValidClass -- OtherClass

another invalid line here!!!

@enduml
"""

# This won't raise an error
result = lenient_parser.parse(uml_with_errors)
print(f"Parsed: {result.summary()}")
print(f"\nWarnings ({len(lenient_parser.warnings)}):")
for w in lenient_parser.warnings:
    print(f"  - {w}")

## Summary

The PlantUML parser:

1. **Input**: Takes a PlantUML string with `@startuml`/`@enduml` markers
2. **Output**: Returns a `ParsedModel` with structured data
3. **Handles**: Classes, enums, all relationship types, cardinalities, labels, notes
4. **Normalizes**: Relationship directions (parent/whole always as source)
5. **Modes**: Strict (raises on errors) or lenient (skips with warnings)

### Key Methods

```python
parser = PlantUMLParser(strict=True)
model = parser.parse(uml_string)

# Access parsed data
model.classes          # List of ParsedClass
model.enums            # List of ParsedEnum  
model.relationships    # List of ParsedRelationship
model.notes            # List of ParsedNote
model.implicit_classes # List of class names only in relationships

# Convenience methods
model.all_class_names  # All class names (explicit + implicit)
model.all_enum_names   # All enum names (standalone + nested)
model.get_class('Name')  # Find a specific class
model.get_enum('Name')   # Find a specific enum
model.summary()        # Get summary string
```